# **Logistic Regression with PySpark**



[AUC- Area Under the Curve](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc#area_under_the_curve_auc)

## **Project 1 - titanic data**

• `PassengerId`: Unique identifier for each passenger.

• `Survived`: Target variable indicating survival status (0 = Did not survive, 1 = Survived).

• `Pclass`: Ticket/socio-economic class (1 = 1st/Upper, 2 = 2nd/Middle, 3 = 3rd/Lower).

• `Name`: Full name of the passenger, including titles and maiden names.

• `Sex`: Biological gender (male or female).

• `Age`: Age in years (fractional for infants under one year; contains missing values).

• `SibSp`: Number of siblings or spouses aboard the ship.

• `Parch`: Number of parents or children aboard the ship.

• `Ticket`: Alphanumeric ticket number.

• `Fare`: Passenger ticket price.

• `Cabin`: Cabin room number (frequently missing/null).

• `Embarked`: Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Titanic").getOrCreate()

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

# 1. Load data
titanic_df = spark.read.csv("titanic.csv",header=True,inferSchema=True)

# 2. Select model columns and remove rows with missing values
model_columns = ["Survived", "Pclass", "Sex", "Age","SibSp", "Parch", "Fare", "Embarked"]

titanic_data = titanic_df.select(*model_columns).na.drop()

# 3. Define preprocessing and model stages
sex_indexer = StringIndexer(inputCol="Sex",outputCol="SexIndex",handleInvalid="keep")

embarked_indexer = StringIndexer(inputCol="Embarked",outputCol="EmbarkedIndex",handleInvalid="keep")

encoder = OneHotEncoder(inputCols=["SexIndex", "EmbarkedIndex"],outputCols=["SexVec", "EmbarkedVec"])

assembler = VectorAssembler(inputCols=["Pclass", "SexVec", "EmbarkedVec","Age", "SibSp", "Parch", "Fare"],outputCol="features")

logistic_regression_titanic = LogisticRegression(featuresCol="features",labelCol="Survived")

titanic_pipeline = Pipeline(stages=[
    sex_indexer,embarked_indexer,
    encoder,assembler,
    logistic_regression_titanic
])

# 4. Split and train
train_titanic, test_titanic = titanic_data.randomSplit([0.75, 0.25])

titanic_model = titanic_pipeline.fit(train_titanic)
predictions = titanic_model.transform(test_titanic)

# 5. Inspect predictions and evaluate
predictions.select("Survived", "probability", "prediction").show()

evaluator = BinaryClassificationEvaluator(labelCol="Survived",rawPredictionCol="rawPrediction",metricName="areaUnderROC")

print(f"Test AUC: {evaluator.evaluate(predictions):.3f}")

+--------+--------------------+----------+
|Survived|         probability|prediction|
+--------+--------------------+----------+
|       0|[0.04747130812921...|       1.0|
|       0|[0.25434733694788...|       1.0|
|       0|[0.20763805815470...|       1.0|
|       0|[0.52948463427339...|       0.0|
|       0|[0.33820239481011...|       1.0|
|       0|[0.50122755865436...|       0.0|
|       0|[0.58160256431934...|       0.0|
|       0|[0.65080305950697...|       0.0|
|       0|[0.66963582729291...|       0.0|
|       0|[0.65952163962586...|       0.0|
|       0|[0.80689220843629...|       0.0|
|       0|[0.81675630885799...|       0.0|
|       0|[0.83212373037678...|       0.0|
|       0|[0.80994590545918...|       0.0|
|       0|[0.10478478451656...|       1.0|
|       0|[0.16698613045212...|       1.0|
|       0|[0.17201543754999...|       1.0|
|       0|[0.55550455247213...|       0.0|
|       0|[0.58737696676585...|       0.0|
|       0|[0.58675227131582...|       0.0|
+--------+-

##  **Project 2 - customer churn**

• `Name` : Name of the latest contact at Company

• `Age`: Customer Age

• `Total_Purchase`: Total Ads Purchased
    
• `Account_Manager`: Binary 0=No manager, 1= Account manager assigned

• `Years`: Totaly Years as a customer

• `Num_sites`: Number of websites that use the service.

• `Onboard_date`: Date that the name of the latest contact was onboarded

• `Location`: Client HQ Address

• `Company`: Name of Client Company


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Churn").getOrCreate()

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

churn_df = spark.read.csv( "customer_churn.csv",header=True,inferSchema=True)

feature_columns = ["Age","Total_Purchase","Account_Manager","Years","Num_Sites"]

assembler = VectorAssembler(inputCols=feature_columns,outputCol="features")

logistic_regression_churn = LogisticRegression(featuresCol="features",labelCol="Churn")

churn_pipeline = Pipeline(stages=[
    assembler,
    logistic_regression_churn
])

train_churn, test_churn = churn_df.randomSplit([0.75, 0.25])

churn_model = churn_pipeline.fit(train_churn)
predictions = churn_model.transform(test_churn)

predictions.select("Churn", "probability", "prediction").show()

evaluator = BinaryClassificationEvaluator(labelCol="Churn",rawPredictionCol="rawPrediction",metricName="areaUnderROC")

print(f"Test AUC: {evaluator.evaluate(predictions):.3f}")

+-----+--------------------+----------+
|Churn|         probability|prediction|
+-----+--------------------+----------+
|    0|[0.99900187744856...|       0.0|
|    1|[0.10492676278148...|       1.0|
|    0|[0.97572426438853...|       0.0|
|    0|[0.98925764838023...|       0.0|
|    0|[0.91870460546211...|       0.0|
|    0|[0.97369673484016...|       0.0|
|    0|[0.91981100427007...|       0.0|
|    0|[0.99214589284436...|       0.0|
|    1|[0.62098611794404...|       0.0|
|    0|[0.61000157536445...|       0.0|
|    0|[0.89052970231845...|       0.0|
|    0|[0.62838282581741...|       0.0|
|    0|[0.98753514733732...|       0.0|
|    0|[0.99472711556611...|       0.0|
|    0|[0.99948944521509...|       0.0|
|    0|[0.13411634648861...|       1.0|
|    0|[0.78486273224331...|       0.0|
|    0|[0.61383906375410...|       0.0|
|    0|[0.93791226109356...|       0.0|
|    0|[0.99985106823097...|       0.0|
+-----+--------------------+----------+
only showing top 20 rows
Test AUC: 0.900

In [3]:
# For new customers, apply the fitted pipeline directly:

final_churn_model = churn_pipeline.fit(churn_df)

new_customers = spark.read.csv("new_customers.csv",header=True,inferSchema=True)

final_results = final_churn_model.transform(new_customers)

final_results.select("Names", "Company", "probability", "prediction").show(truncate=False)

+--------------+----------------+------------------------------------------+----------+
|Names         |Company         |probability                               |prediction|
+--------------+----------------+------------------------------------------+----------+
|Andrew Mccall |King Ltd        |[0.9021801592176474,0.09781984078235262]  |0.0       |
|Michele Wright|Cannon-Benson   |[0.0019838025978421254,0.9980161974021579]|1.0       |
|Jeremy Chang  |Barron-Robertson|[0.022551133124332524,0.9774488668756675] |1.0       |
|Megan Ferguson|Sexton-Golden   |[0.006086220767149694,0.9939137792328503] |1.0       |
|Taylor Young  |Wood LLC        |[0.751150561448994,0.24884943855100605]   |0.0       |
|Jessica Drake |Parks-Robbins   |[0.15582818486369907,0.8441718151363009]  |1.0       |
+--------------+----------------+------------------------------------------+----------+



#

# **Titanic (Project 1) Code Breakdown**

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Titanic").getOrCreate()
df = spark.read.csv('titanic.csv',inferSchema=True,header=True)

In [5]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [6]:
df.describe().show()

+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|summary|      PassengerId|           Survived|            Pclass|                Name|   Sex|               Age|             SibSp|              Parch|            Ticket|             Fare|Cabin|Embarked|
+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|  count|              891|                891|               891|                 891|   891|               714|               891|                891|               891|              891|  204|     889|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642|                NULL|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824|260318.54916792738| 32.20420

In [7]:
df.columns

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [8]:
df.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [9]:
selected_cols = df.select(['Survived','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked'])

**Dealing with nulls - dropping the missing data**

In [10]:
cleaned_data = selected_cols.na.drop()

In [11]:
from pyspark.ml.feature import VectorAssembler, VectorIndexer, OneHotEncoder, StringIndexer

In [12]:
gender_indexer = StringIndexer(inputCol="Sex",outputCol="SexIndex")
gender_encoder = OneHotEncoder(inputCol="SexIndex",outputCol="SexVec")

In [13]:
embark_indexer = StringIndexer(inputCol="Embarked",outputCol="EmbarkIndex")
embark_encoder = OneHotEncoder(inputCol="EmbarkIndex",outputCol="EmbarkVec")

In [14]:
assembler = VectorAssembler(inputCols=["Pclass","SexVec","EmbarkVec","Age","SibSp","Parch","Fare"],
                            outputCol="features")

In [15]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

In [16]:
logistic_regression_titanic = LogisticRegression(featuresCol="features", labelCol="Survived")

In [17]:
pipline = Pipeline(stages=[
    gender_indexer,embark_indexer,
    gender_encoder,embark_encoder,
    assembler, logistic_regression_titanic
])

In [18]:
train_data, test_data = cleaned_data.randomSplit([0.75,0.25])

In [19]:
fit_model = pipline.fit(train_data)

In [20]:
results = fit_model.transform(test_data)

In [21]:
results.show()

+--------+------+------+----+-----+-----+-------+--------+--------+-----------+-------------+-------------+--------------------+--------------------+--------------------+----------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|   Fare|Embarked|SexIndex|EmbarkIndex|       SexVec|    EmbarkVec|            features|       rawPrediction|         probability|prediction|
+--------+------+------+----+-----+-----+-------+--------+--------+-----------+-------------+-------------+--------------------+--------------------+--------------------+----------+
|       0|     1|female|25.0|    1|    2| 151.55|       S|     1.0|        0.0|    (1,[],[])|(2,[0],[1.0])|[1.0,0.0,1.0,0.0,...|[-2.5828513631859...|[0.07025026527090...|       1.0|
|       0|     1|female|50.0|    0|    0|28.7125|       C|     1.0|        1.0|    (1,[],[])|(2,[1],[1.0])|(8,[0,3,4,7],[1.0...|[-2.7952817728531...|[0.05757967316800...|       1.0|
|       0|     1|  male|28.0|    1|    0|82.1708|       C|     0.0|        1.0|(1,[0],[1.0

In [22]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [23]:
eval = BinaryClassificationEvaluator(rawPredictionCol="prediction", labelCol="Survived")
# by default, when test data gets transformed, the predicted column will be automatically named as "prediction"

In [24]:
results.select("Survived","prediction").show()

+--------+----------+
|Survived|prediction|
+--------+----------+
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       1.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
+--------+----------+
only showing top 20 rows


In [25]:
AUC = eval.evaluate(results)

In [26]:
AUC


0.8004464285714286